# 02 — Files, JSON and APIs

### Learning objectives

By the end you will:

- read/write JSON and JSONL;
- distinguish a Python object from a serialized representation;
- consume an HTTP API;
- validate a response before using it;
- create a local fallback fixture for reproducibility;
- normalize nested JSON into tabular records.

### Why this matters

Data engineering often starts here:

**external system → raw bytes → parsed records → validated data → durable format**

The database or warehouse comes later.


## Capability contract

**Capability:** C01 — System Foundations  
**Workstream:** Foundation  
**Primary roles:** Data Engineer|Analytics Engineer  
**You will prove:** Explain the mechanism and reproduce it with a deterministic experiment.


## 🖊️ Sketch note

![Sketch note — Files, JSON & APIs](../assets/sketch-notes/02-files-json-apis.svg)

> **Visual-first rule:** spend 30–60 seconds explaining the sketch in your own words before reading the implementation. Then return to it after the experiment and redraw it from memory.


## Agent-native lens

**Agent can do:** routine implementation or a first-pass analysis.

**You must understand:** the mechanism and its assumptions.

**You must verify:** use tests, measurements or adversarial data that could prove the output wrong.

**You must decide:** the trade-off, failure response or production boundary.


## Apprenticeship bridge

Use this notebook to prepare for the following Acme Commerce work scenario(s). The scenario gives you symptoms and constraints; it does **not** give you the diagnosis.

- **S02 — API schema drift (DATA-1051)** → [`../work_simulations/02_acme_commerce_api_schema_drift/README.md`](../work_simulations/02_acme_commerce_api_schema_drift/README.md)

**How to use the bridge:** finish the experiment and independent challenge here, then enter the ticket without reopening the notebook as an answer key. Bring your own prediction, evidence and verification plan.


## Why this exists

Start with the engineering problem. Know what would go wrong if this mechanism did not exist.


## Predict before you run

Predict which malformed record or API response should be rejected and why.


## Measure

Count accepted, rejected and normalized records.


## Diagnose

Separate source failure, contract drift and transformation bugs.


In [ ]:
%pip -q install "requests>=2.31,<3" "pandas>=2,<4"


In [ ]:
import json
from pathlib import Path
import requests
import pandas as pd

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

print("Data directory:", DATA_DIR.resolve())


## 1. JSON is a representation, not "the data"

A Python dictionary:

```python
{"name": "Ada", "score": 10}
```

can be serialized as JSON text:

```json
{"name": "Ada", "score": 10}
```

The distinction matters because serialization introduces questions about:

- types;
- encoding;
- schema;
- compatibility;
- size;
- parsing cost.


In [ ]:
record = {
    "customer_id": "C001",
    "name": "Ada",
    "tags": ["finance", "data"],
    "active": True,
    "score": 10.5,
}

json_text = json.dumps(record, indent=2)
print(json_text)

round_trip = json.loads(json_text)
assert round_trip == record
print("JSON round-trip: PASS")


## 2. JSONL for event-style data

JSONL (JSON Lines) stores one JSON object per line.

That makes it convenient for:

- event streams;
- logs;
- incremental processing;
- append-oriented datasets.

It also lets us process records one at a time rather than loading one giant JSON array.


In [ ]:
jsonl_path = DATA_DIR / "events.jsonl"

records = [
    {"event_id": 1, "customer_id": "C1", "event": "view", "amount": 0},
    {"event_id": 2, "customer_id": "C1", "event": "purchase", "amount": 120},
    {"event_id": 3, "customer_id": "C2", "event": "purchase", "amount": 80},
]

with jsonl_path.open("w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record) + "\n")

loaded = []
with jsonl_path.open(encoding="utf-8") as f:
    for line in f:
        if line.strip():
            loaded.append(json.loads(line))

assert loaded == records
print(f"Wrote and read {len(loaded)} JSONL records.")


## 3. Consume an API — with a fallback

For a teaching repository, depending on a live API is fragile.

We'll try a public endpoint, but also keep a local fixture so the notebook remains useful if the network is unavailable.

This pattern will recur throughout the course:

**real external source + deterministic fallback**


In [ ]:
API_URL = "https://jsonplaceholder.typicode.com/posts"

try:
    response = requests.get(API_URL, timeout=10)
    response.raise_for_status()
    api_records = response.json()
    source = "live API"
except Exception as exc:
    print("Live API unavailable; using local fixture.")
    print(type(exc).__name__, str(exc)[:120])
    api_records = [
        {"userId": 1, "id": 1, "title": "local fixture", "body": "offline-safe example"},
        {"userId": 2, "id": 2, "title": "second fixture", "body": "reproducible teaching data"},
    ]
    source = "local fixture"

print("Source:", source)
print("Records:", len(api_records))
assert isinstance(api_records, list)
assert all(isinstance(x, dict) for x in api_records)


## 4. Validate before transforming

Never assume an API response has the shape you expected.

A tiny validator makes the assumption explicit.


In [ ]:
def validate_post(record: dict) -> None:
    required = {"userId", "id", "title", "body"}
    missing = required - record.keys()
    if missing:
        raise ValueError(f"Missing fields: {sorted(missing)}")

for record in api_records[:5]:
    validate_post(record)

print("API shape checks: PASS")


## 5. Normalize nested/semi-structured data

Once data is validated, we can move toward tabular processing.

The goal isn't to erase structure blindly. It is to create a representation appropriate for the next computation.


In [ ]:
posts = pd.DataFrame(api_records)

display(posts.head())
print("Columns:", list(posts.columns))
print("Shape:", posts.shape)


## 6. Break it

Imagine an API changes `title` to `headline`.

What should happen?

A mature pipeline should not silently produce a table full of nulls. It should detect the contract change.

Try modifying one record below and rerun validation.


In [ ]:
# Exercise: uncomment the next two lines and observe the failure.
# broken = dict(api_records[0])
# broken["title"] = broken.pop("headline")

# Then write a validator that also checks:
# - id is an integer
# - title is non-empty
# - body is a string


## Engineering challenge

Create a function:

```python
fetch_and_normalize_posts(url)
```

Requirements:

1. fetch with a timeout;
2. raise a useful error for an HTTP failure;
3. validate the top-level response;
4. return a DataFrame;
5. never silently convert a malformed response into an apparently valid dataset.

**Production connection:** this is the beginning of an ingestion boundary. Later we will add schemas, retries, idempotency, quality checks, incremental loads, lineage and orchestration.
